In [1]:
// Añadimos Apache Spark Core y SQL para Scala 2.13
// El sufijo _2.13 es gestionado automáticamente por $ivy cuando usamos %%
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

println("✅ Dependencias de Spark cargadas correctamente")

Downloaded https://repo1.maven.org/maven2/com/google/protobuf/protobuf-java/4.28.2/protobuf-java-4.28.2.pom
Downloaded https://repo1.maven.org/maven2/com/google/protobuf/protobuf-parent/4.28.2/protobuf-parent-4.28.2.pom
Downloaded https://repo1.maven.org/maven2/com/google/protobuf/protobuf-bom/4.28.2/protobuf-bom-4.28.2.pom


✅ Dependencias de Spark cargadas correctamente


import $ivy.$
import $ivy.$

In [2]:
import org.apache.log4j.{Level, Logger}

// Silenciamos los logs de Spark para que el output sea legible
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

println("✅ Logs de Spark configurados (solo se mostrarán errores)")

✅ Logs de Spark configurados (solo se mostrarán errores)


import org.apache.log4j.{Level, Logger}

La SparkSession es el punto de entrada unificado de Spark desde la versión 2.0. Reemplaza al antiguo SparkContext y al SQLContext. En la práctica, siempre empezarás creando una.

In [3]:
import org.apache.spark.sql.SparkSession

// Creamos la SparkSession en modo local
// "local[*]" significa: usa todos los núcleos de CPU disponibles
val spark = SparkSession.builder()
  .appName("IntroSpark")   // nombre visible en el Spark UI
  .master("local[*]")                    // modo local, todos los núcleos
  .config("spark.ui.showConsoleProgress", "false")  // sin barras de progreso
  .getOrCreate()

// Obtenemos el SparkContext desde la sesión
val sc = spark.sparkContext

println(s"✅ SparkSession creada correctamente")
println(s"   Versión de Spark: ${spark.version}")
println(s"   Nombre de la app: ${sc.appName}")
println(s"   Master:           ${sc.master}")
println(s"   Spark UI:         http://localhost:4040")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 11:10:53 INFO SparkContext: Running Spark version 4.1.1
26/04/21 11:10:53 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/21 11:10:53 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/21 11:10:54 INFO ResourceUtils: ==============================================================
26/04/21 11:10:54 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/21 11:10:54 INFO ResourceUtils: ==============================================================
26/04/21 11:10:54 INFO SparkContext: Submitted application: IntroSpark
26/04/21 11:10:54 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/21 11:10:54 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/21 11:10:54 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/21 11:10:54 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/21 11:10:54 INFO SecurityManager: Changing view acls to

2026-04-21T09:10:54.649199200Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3851d6ed
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@241c2f5b

Crear el primer RDD
Un RDD se puede crear de dos formas: paralelizando una colección existente en memoria (como haremos ahora, ideal para aprender) o leyendo datos de una fuente externa como ficheros de texto, HDFS o S3 (lo haremos en sesiones posteriores)

In [4]:
// Creamos un RDD a partir de una lista de Scala
// parallelize() distribuye la colección en particiones
val numeros = sc.parallelize(List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10))

// Inspeccionamos el RDD (esto NO es una acción todavía)
println(s"Tipo del RDD:       ${numeros.getClass.getSimpleName}")
println(s"Número de partes:   ${numeros.getNumPartitions}")
println(s"¿Se ha ejecutado?:  No — solo hemos definido el plan")

Tipo del RDD:       ParallelCollectionRDD
Número de partes:   8
¿Se ha ejecutado?:  No — solo hemos definido el plan


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[0] at parallelize at cmd4.sc:3

 Primera transformación: filter y map
Ahora aplicamos transformaciones. Recuerda: estas líneas no calculan nada todavía, solo construyen el plan de ejecución.

In [5]:
// Transformación 1: filtrar solo los números pares
// Esta línea NO ejecuta nada — solo define el plan
val pares = numeros.filter(n => n % 2 == 0)

// Transformación 2: calcular el cuadrado de cada número par
// Tampoco ejecuta nada — extiende el plan anterior
val cuadradosDePares = pares.map(n => n * n)

println("Transformaciones definidas (plan listo, pero aún sin ejecutar):")
println(s"  numeros            → ${numeros.toDebugString.split('\n').head}")
println(s"  pares              → filtrar pares")
println(s"  cuadradosDePares   → elevar al cuadrado")
println()
println("Para ejecutar el plan, necesitamos una ACCIÓN...")

Transformaciones definidas (plan listo, pero aún sin ejecutar):
  numeros            → (8) ParallelCollectionRDD[0] at parallelize at cmd4.sc:3 []
  pares              → filtrar pares
  cuadradosDePares   → elevar al cuadrado

Para ejecutar el plan, necesitamos una ACCIÓN...


pares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[1] at filter at cmd5.sc:3
cuadradosDePares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[2] at map at cmd5.sc:7

Primera acción: collect()
Aquí es donde ocurre la magia. Al llamar a collect(), Spark optimiza el plan completo, lo distribuye entre los executors y devuelve el resultado al Driver.

In [6]:
// collect() es una ACCIÓN — aquí sí se ejecuta todo el plan
// ⚠️ En producción, collect() solo se usa con datasets pequeños
//    porque trae todos los datos al Driver (memoria del notebook)
val resultado = cuadradosDePares.collect()

println("✅ Acción collect() ejecutada — el job ha terminado")
println(s"Cuadrados de los números pares del 1 al 10:")
println(resultado.mkString(", "))

✅ Acción collect() ejecutada — el job ha terminado
Cuadrados de los números pares del 1 al 10:
4, 16, 36, 64, 100


resultado: Array[Int] = Array(4, 16, 36, 64, 100)

 Acciones de agregación: count, sum, reduce
Practicamos las acciones de agregación más comunes:

In [7]:
val datos = sc.parallelize(List(10, 20, 30, 40, 50, 60, 70, 80, 90, 100))

// count() devuelve el número de elementos (una acción)
val total = datos.count()

// sum() calcula la suma de todos los elementos
val suma = datos.sum()

// reduce() aplica una función binaria de forma distribuida
// Aquí calculamos el máximo: de dos elementos, quedamos con el mayor
val maximo = datos.reduce((a, b) => if (a > b) a else b)

// first() devuelve el primer elemento (sin traer todo al Driver)
val primero = datos.first()

println(s"Número de elementos: $total")
println(s"Suma total:          $suma")
println(s"Valor máximo:        $maximo")
println(s"Primer elemento:     $primero")

Número de elementos: 10
Suma total:          550.0
Valor máximo:        100
Primer elemento:     10


datos: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[3] at parallelize at cmd7.sc:1
total: Long = 10L
suma: Double = 550.0
maximo: Int = 100
primero: Int = 10

 Transformaciones sobre texto: el WordCount clásico
El "Hola Mundo" de Spark es el WordCount: contar cuántas veces aparece cada palabra en un texto. Es sencillo pero ilustra perfectamente el modelo map-reduce distribuido

In [10]:
import spark.implicits._

val df = Seq(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
).toDF("linea")

val palabras = df
  .flatMap(row => row.getString(0).split(" "))
  .toDF("palabra")

val conteo = palabras
  .groupBy("palabra")
  .count()
  .orderBy($"count".desc)

conteo.show(10)

+-------------+-----+
|      palabra|count|
+-------------+-----+
|        spark|    4|
|           de|    3|
|           es|    2|
|       apache|    2|
|           en|    2|
|        scala|    2|
|procesamiento|    1|
|  distribuido|    1|
|        motor|    1|
|           un|    1|
+-------------+-----+
only showing top 10 rows


import spark.implicits._
df: org.apache.spark.sql.package.DataFrame = [linea: string]
palabras: org.apache.spark.sql.package.DataFrame = [palabra: string]
conteo: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [palabra: string, count: bigint]

 Caché: guardando resultados en memoria
Una de las grandes ventajas de Spark sobre MapReduce es la capacidad de guardar un RDD en memoria para reutilizarlo sin recalcularlo. Esto es especialmente útil cuando el mismo RDD se usa en múltiples acciones.

In [11]:
import org.apache.spark.storage.StorageLevel

// RDD que simula un dataset de ventas con procesamiento costoso
val ventas = sc.parallelize(1 to 1_000_000)
  .map(id => (id, scala.util.Random.nextInt(1000), scala.util.Random.nextInt(50)))
  // (id, importe, region_id)

// Indicamos a Spark que guarde este RDD en memoria cuando se calcule por primera vez
// MEMORY_AND_DISK: si no cabe en RAM, desborda a disco (más seguro)
ventas.persist(StorageLevel.MEMORY_AND_DISK)

// Primera acción: Spark calcula el RDD y lo guarda en caché
val totalVentas = ventas.count()
println(s"Total de ventas: $totalVentas")

// Segunda acción: Spark REUTILIZA el RDD cacheado (mucho más rápido)
val importeTotal = ventas.map { case (_, importe, _) => importe }.sum()
println(f"Importe total: $importeTotal%.0f")

// Cuando ya no necesitamos el RDD, liberamos la memoria
ventas.unpersist()
println("✅ Caché liberado")

Total de ventas: 1000000
Importe total: 499306198
✅ Caché liberado


import org.apache.spark.storage.StorageLevel
ventas: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[30] at map at cmd11.sc:5
res11_2: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[30] at map at cmd11.sc:5
totalVentas: Long = 1000000L
importeTotal: Double = 4.99306198E8
res11_7: org.apache.spark.rdd.RDD[(Int, Int, Int)] = MapPartitionsRDD[30] at map at cmd11.sc:5

El DAG: visualizando el plan de ejecución
Spark organiza el trabajo en un DAG (Directed Acyclic Graph, Grafo Acíclico Dirigido). Cada transformación añade un nodo al grafo; cada acción dispara la ejecución del subgrafo completo. El método toDebugString() muestra el linaje de un RDD:

In [12]:
// Construimos un pipeline de varias transformaciones
val pipeline = sc.parallelize(1 to 20)
  .filter(_ % 2 == 0)           // solo pares
  .map(n => n * n)              // elevar al cuadrado
  .filter(_ > 50)               // solo los mayores que 50
  .map(n => s"resultado: $n")   // convertir a string

// toDebugString muestra el grafo de dependencias (linaje del RDD)
println("Linaje del RDD (de abajo a arriba = del origen al resultado):")
println("─" * 60)
println(pipeline.toDebugString)
println("─" * 60)
println()

// Ejecutamos la acción para ver el resultado
val resultados = pipeline.collect()
println(s"Resultados (${resultados.length} elementos):")
resultados.foreach(println)

Linaje del RDD (de abajo a arriba = del origen al resultado):
────────────────────────────────────────────────────────────
(8) MapPartitionsRDD[37] at map at cmd12.sc:6 []
 |  MapPartitionsRDD[36] at filter at cmd12.sc:5 []
 |  MapPartitionsRDD[35] at map at cmd12.sc:4 []
 |  MapPartitionsRDD[34] at filter at cmd12.sc:3 []
 |  ParallelCollectionRDD[33] at parallelize at cmd12.sc:2 []
────────────────────────────────────────────────────────────

Resultados (7 elementos):
resultado: 64
resultado: 100
resultado: 144
resultado: 196
resultado: 256
resultado: 324
resultado: 400


pipeline: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[37] at map at cmd12.sc:6
resultados: Array[String] = Array(
  "resultado: 64",
  "resultado: 100",
  "resultado: 144",
  "resultado: 196",
  "resultado: 256",
  "resultado: 324",
  "resultado: 400"
)

Celda 11 — Cerrar la SparkSession
Es buena práctica cerrar la SparkSession al terminar. Al cerrarla, el Spark UI (puerto 4040) también se apaga.

In [13]:
spark.stop()
println("✅ SparkSession cerrada correctamente")
println("   El Spark UI (localhost:4040) ya no está disponible")

2026-04-21T09:21:56.925599600Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

Ejercicios propuestos:
📌 Todos los ejercicios se resuelven en el mismo notebook. Recuerda cargar las dependencias y crear la SparkSession en las primeras celdas antes de empezar. La SparkSession solo necesitas crearla una vez por sesión de notebook.

⚙️ Celda de configuración inicial (ejecutar antes de los ejercicios)

In [14]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Ejercicios")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext
println(s"✅ Entorno listo — Spark ${spark.version}")

✅ Entorno listo — Spark 4.1.1


import $ivy.$
import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@100aea95
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@4f1418ee

 Ejercicios
Ejercicio 1 — Crear y explorar un RDD de temperaturas
Crea un RDD a partir de la siguiente lista de temperaturas en grados Celsius registradas durante una semana en distintas ciudades. Luego imprime cuántas particiones tiene el RDD y confirma de qué tipo es usando getClass.getSimpleName.

In [15]:
val temperaturas = List(22.5, 18.0, 35.1, 12.3, 28.7, 9.4, 31.0, 25.5, 17.2, 33.8)
val temperaturasPar = sc.parallelize(List(22.5, 18.0, 35.1, 12.3, 28.7, 9.4, 31.0, 25.5, 17.2, 33.8))
//cuantas particiones tiene
println(s"Número de partes:   ${temperaturasPar.getNumPartitions}")
// que tipo es?
println(s"Tipo del RDD:       ${temperaturasPar.getClass.getSimpleName}")

Número de partes:   8
Tipo del RDD:       ParallelCollectionRDD


temperaturas: List[Double] = List(
  22.5,
  18.0,
  35.1,
  12.3,
  28.7,
  9.4,
  31.0,
  25.5,
  17.2,
  33.8
)
temperaturasPar: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[0] at parallelize at cmd15.sc:2

Ejercicio 2 — Filtrar temperaturas extremas
Usando el RDD del ejercicio anterior, aplica dos transformaciones encadenadas: primero filtra las temperaturas superiores a 20 grados, y sobre ese resultado filtra de nuevo las que sean inferiores a 30 grados. Recoge el resultado con collect() e imprímelo.

In [16]:
val tempMas20 = temperaturasPar.filter(t => t>20)
val tempMen30 = tempMas20.filter(te => te<30)
val resultado = tempMen30.collect()
println(resultado.mkString(", "))

22.5, 28.7, 25.5


tempMas20: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[1] at filter at cmd16.sc:1
tempMen30: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[2] at filter at cmd16.sc:2
resultado: Array[Double] = Array(22.5, 28.7, 25.5)

Ejercicio 3 — Transformación de unidades
A partir del mismo RDD de temperaturas, crea un nuevo RDD que convierta cada valor de Celsius a Fahrenheit usando la fórmula F = C * 9/5 + 32. Recoge el resultado e imprímelo redondeado a un decimal.

In [22]:
val celAFah = temperaturasPar.map(t => t*9/5 + 32)
val cambioTemp = celAFah.collect().map(c => f"$c%.1f")
println(f"Temperaturas en Fahrenheit: " + cambioTemp.mkString(", "))

Temperaturas en Fahrenheit: 72,5, 64,4, 95,2, 54,1, 83,7, 48,9, 87,8, 77,9, 63,0, 92,8


celAFah: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[8] at map at cmd22.sc:1
cambioTemp: Array[String] = Array(
  "72,5",
  "64,4",
  "95,2",
  "54,1",
  "83,7",
  "48,9",
  "87,8",
  "77,9",
  "63,0",
  "92,8"
)

Ejercicio 4 — Acciones de estadística básica
Crea un RDD con los precios de los siguientes artículos de una tienda online y calcula, usando las acciones de Spark, el número total de artículos, el precio más alto y el precio más bajo. No uses collect() para esto: encuentra las acciones específicas que devuelven cada valor directamente.

In [23]:
val precios = sc.parallelize(List(49.99, 12.50, 199.00, 7.99, 89.90, 34.99, 149.95, 22.00))
// Tu código aquí

precios: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[9] at parallelize at cmd23.sc:1

In [29]:
println(f"Numero de articulos: " + precios.count())
println(f"Precio más alto: " + precios.max())
println(f"Precio más bajo: " + precios.min())

Numero de articulos: 8
Precio más alto: 199.0
Precio más bajo: 7.99


Ejercicio 5 — Entender la lazy evaluation
Este ejercicio es conceptual y de observación. Ejecuta las siguientes dos celdas por separado y observa en qué momento aparece actividad en el Spark UI (http://localhost:4040).

Celda A — solo transformaciones:

In [30]:
val numeros = sc.parallelize(1 to 100)
val multiplosde7 = numeros.filter(n => n % 7 == 0)
val dobles = multiplosde7.map(n => n * 2)
println("Celda A ejecutada — ¿Hay un nuevo job en el Spark UI?")

Celda A ejecutada — ¿Hay un nuevo job en el Spark UI?


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[13] at parallelize at cmd30.sc:1
multiplosde7: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[14] at filter at cmd30.sc:2
dobles: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[15] at map at cmd30.sc:3

In [31]:
val resultado = dobles.collect()
println("Celda B ejecutada — ¿Y ahora en el Spark UI?")
println(s"Resultado: ${resultado.mkString(", ")}")

Celda B ejecutada — ¿Y ahora en el Spark UI?
Resultado: 14, 28, 42, 56, 70, 84, 98, 112, 126, 140, 154, 168, 182, 196


resultado: Array[Int] = Array(
  14,
  28,
  42,
  56,
  70,
  84,
  98,
  112,
  126,
  140,
  154,
  168,
  182,
  196
)

💭 Reflexiona: ¿En qué celda apareció el job en el Spark UI? ¿Qué nos dice eso sobre cuándo Spark realmente trabaja?
R: Porque al hacer collect() es cunado de verdad spark ejecuta una accion, lo demas anterior solo son filtrados que solo hace de manera superficial.

Ejercicio 6 — Pipeline de transformaciones sobre nombres
Crea un RDD a partir de la siguiente lista de nombres de empleados. Aplica en cadena estas tres transformaciones: convierte cada nombre a mayúsculas, filtra solo los que empiecen por la letra "A", y luego ordénalos alfabéticamente. Usa collect() y muestra el resultado.

In [45]:
val empleados = sc.parallelize(List(
  "ana garcía", "pedro López", "alba martínez", "carlos ruiz",
  "adriana vega", "beatriz soler", "antonio mora", "sara jiménez"
))
// Tu código aquí

empleados: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[95] at parallelize at cmd45.sc:1

In [48]:
val transformacion = empleados
 .map(e => e.toUpperCase)
 .filter(e => e.startsWith("A"))
 .collect()
 .sorted

println("Empleados cuyo nombre empieza por A: " + transformacion.mkString(", "))

Empleados cuyo nombre empieza por A: ADRIANA VEGA, ALBA MARTÍNEZ, ANA GARCÍA, ANTONIO MORA


transformacion: Array[String] = Array(
  "ADRIANA VEGA",
  "ALBA MARTÍNEZ",
  "ANA GARCÍA",
  "ANTONIO MORA"
)

Ejercicio 7 — flatMap sobre líneas de texto
Dado el siguiente RDD de frases sobre tecnología, usa flatMap para obtener un RDD con todas las palabras individuales de todas las frases. Luego cuenta cuántas palabras hay en total usando una acción.

In [49]:
val frases = sc.parallelize(List(
  "spark procesa datos a gran velocidad",
  "scala es el lenguaje nativo de spark",
  "los datos se procesan en memoria ram",
  "hadoop almacena datos en disco hdfs"
))
// Tu código aquí

frases: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[107] at parallelize at cmd49.sc:1

In [52]:
val palabras = frases.flatMap(f =>f.split(" "))
val total = palabras.count()

println("Total de palabras en todas las frases: " + total)

Total de palabras en todas las frases: 26


palabras: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[110] at flatMap at cmd52.sc:1
total: Long = 26L

Ejercicio 8 — reduce para encontrar el máximo sin usar max()
Crea un RDD con los siguientes valores de ventas diarias (en euros) y usa reduce() para encontrar el valor más alto. No está permitido usar los métodos max() ni sum() de Spark: el cálculo debe hacerse con una función lambda dentro de reduce.

In [53]:
val ventasDiarias = sc.parallelize(List(1520.0, 890.5, 2340.0, 670.0, 1890.75, 3100.0, 450.25))

ventasDiarias: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[111] at parallelize at cmd53.sc:1

In [54]:
val maxi = ventasDiarias.reduce((a,b)=> if (a>b) a else b)
println("Venta diaria mas alta: " + maxi)


Venta diaria mas alta: 3100.0


maxi: Double = 3100.0

Ejercicio 9 — Contar elementos que cumplen una condición
A partir de un RDD con las edades de los visitantes de una web, calcula cuántos visitantes son mayores de edad (18 años o más) usando una combinación de filter y count(). Luego calcula también el porcentaje que representan sobre el total.

In [55]:
val edades = sc.parallelize(List(
  14, 23, 17, 35, 16, 28, 42, 15, 19, 31, 13, 25, 18, 22, 16, 45, 20, 12
))
// Tu código aquí

edades: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[112] at parallelize at cmd55.sc:1

In [61]:
val mayores = edades.filter(e => e>=18).count()
val totales = edades.count()
val porcentaje = (mayores.toDouble / total) * 100

println("Total visitantes: " + edades.count())
println("Mayores de edad: " + mayores)
println(f"Porcentaje: $porcentaje%.2f %%")

Total visitantes: 18
Mayores de edad: 11
Porcentaje: 42,31 %


mayores: Long = 11L
totales: Long = 18L
porcentaje: Double = 42.30769230769231

Ejercicio 10 — WordCount de hashtags
Tienes el siguiente RDD con publicaciones de redes sociales. Usa flatMap, map y reduceByKey para contar cuántas veces aparece cada hashtag (palabras que empiezan por #). Muestra los resultados ordenados de mayor a menor frecuencia.

In [62]:
val publicaciones = sc.parallelize(List(
  "hoy aprendemos #spark con #scala en clase",
  "me encanta #scala es un lenguaje potente",
  "procesando big data con #spark y #hadoop",
  "#spark es más rápido que #hadoop en memoria",
  "aprendiendo #scala y #spark juntos hoy"
))
// Pista: después de flatMap, filtra solo las palabras que empiecen por "#"
// Tu código aquí

publicaciones: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[119] at parallelize at cmd62.sc:1


Ejercicio 11 — Verificar el linaje con toDebugString
Construye el siguiente pipeline de transformaciones y usa toDebugString para imprimir su linaje antes de ejecutar ninguna acción. Luego ejecuta collect() y comprueba que el resultado es correcto.

In [63]:
val notas = sc.parallelize(List(4.5, 7.0, 3.2, 9.1, 5.5, 8.8, 2.0, 6.3, 7.7, 4.9))
// Paso 1: filtra las notas aprobadas (>= 5.0)
// Paso 2: multiplica cada nota por 10 para obtener la puntuación sobre 100
// Paso 3: filtra las que sean mayores o iguales a 70 (notable o superior)
// Tu código aquí — imprime toDebugString antes de collect()

notas: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[120] at parallelize at cmd63.sc:1

Ejercicio 12 — take vs collect
Crea un RDD con los números del 1 al 1000 y aplica una transformación que calcule el cubo de cada número (n * n * n). Luego usa take(5) para mostrar solo los cinco primeros resultados sin traer los 1000 elementos al Driver. Como segundo paso, usa count() para confirmar cuántos elementos tiene el RDD resultante.

In [64]:
val rango = sc.parallelize(1 to 1000)
// Tu código aquí

rango: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[121] at parallelize at cmd64.sc:1


Ejercicio 13 — Cachear un RDD reutilizado
Crea un RDD con los números del 1 al 500.000. Aplica un filter que se quede solo con los múltiplos de 3. Persiste ese RDD en memoria antes de ejecutar ninguna acción. Luego realiza dos acciones consecutivas sobre él: count() y sum(). Finalmente, libera el caché con unpersist().

El objetivo es observar en el Spark UI que la segunda acción reutiliza el RDD cacheado en lugar de recalcularlo.

In [ ]:
val granRdd = sc.parallelize(1 to 500000)
// Tu código aquí:
// 1. Crea el RDD de múltiplos de 3
// 2. Persiste el RDD
// 3. count() — primera acción
// 4. sum()   — segunda acción (debe usar el caché)
// 5. unpersist()

Ejercicio 14 — Crear y nombrar correctamente la SparkSession
Sin cerrar la sesión que ya tienes abierta, observa el comportamiento de getOrCreate() cuando la sesión ya existe. Intenta crear una segunda SparkSession con un appName diferente y comprueba qué nombre devuelve en realidad.

In [65]:
// Intenta crear una SEGUNDA SparkSession con otro nombre
val spark2 = SparkSession.builder()
  .appName("SesionNueva-Que-No-Existira")
  .master("local[*]")
  .getOrCreate()

// ¿Cuál es el nombre real de spark2?
println(s"Nombre de spark2: ${spark2.sparkContext.appName}")
println(s"¿spark y spark2 son la misma sesión?: ${spark eq spark2}")

Nombre de spark2: Ejercicios
¿spark y spark2 son la misma sesión?: true


spark2: SparkSession = org.apache.spark.sql.classic.SparkSession@100aea95

Ejercicio 15 — Pipeline completo: de datos en bruto a resumen
Este ejercicio integra todo lo visto en la sesión. Tienes el siguiente RDD con registros de accesos a un servidor web (formato: "IP MÉTODO RUTA CÓDIGO_HTTP"). Construye un pipeline que:

Filtre solo los registros con código HTTP 404 (página no encontrada).
Extraiga únicamente la ruta de cada registro fallido.
Cuente cuántas veces aparece cada ruta con reduceByKey.
Ordene las rutas por número de accesos fallidos de mayor a menor.
Muestre el resultado con take(5).

In [ ]:
val logs = sc.parallelize(List(
  "192.168.1.1 GET /inicio 200",
  "192.168.1.2 GET /productos 404",
  "192.168.1.3 POST /login 200",
  "192.168.1.4 GET /productos 404",
  "192.168.1.5 GET /contacto 404",
  "192.168.1.6 GET /inicio 200",
  "192.168.1.7 GET /productos 404",
  "192.168.1.8 GET /blog 404",
  "192.168.1.9 GET /contacto 404",
  "192.168.1.10 GET /productos 404"
))

// Pista: cada línea se puede dividir con .split(" ")
// El código HTTP es el último elemento del array resultante (índice 3)
// La ruta es el tercer elemento (índice 2)
// Tu código aquí

📋 Los datos
El equipo de ingeniería te ha proporcionado los datos del último día de actividad en formato de lista Scala. Cada elemento representa una reproducción y contiene: el identificador del usuario, el título del contenido, el género y la duración visualizada en minutos.

El formato de cada registro es una tupla de cuatro campos: (usuario_id, titulo, genero, minutos_vistos)

In [66]:
val reproducciones = List(
  ("U001", "La Casa de Papel",         "Serie",        47),
  ("U002", "Dune: Parte Dos",          "Película",     95),
  ("U003", "Cosmos: Mundos Posibles",  "Documental",   42),
  ("U001", "Dune: Parte Dos",          "Película",    107),
  ("U004", "La Casa de Papel",         "Serie",        52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie",   58),
  ("U002", "Cosmos: Mundos Posibles",  "Documental",   38),
  ("U006", "La Casa de Papel",         "Serie",        49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie",   61),
  ("U007", "Dune: Parte Dos",          "Película",     88),
  ("U005", "La Casa de Papel",         "Serie",        44),
  ("U008", "Cosmos: Mundos Posibles",  "Documental",   55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie",   63),
  ("U009", "Dune: Parte Dos",          "Película",    112),
  ("U006", "Cosmos: Mundos Posibles",  "Documental",   29),
  ("U010", "La Casa de Papel",         "Serie",        51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie",   57),
  ("U008", "La Casa de Papel",         "Serie",        46),
  ("U001", "Cosmos: Mundos Posibles",  "Documental",   33),
  ("U009", "La Casa de Papel",         "Serie",        48),
  ("U010", "Dune: Parte Dos",          "Película",    102),
  ("U002", "El Problema de los 3 Cuerpos", "Serie",   60),
  ("U003", "Dune: Parte Dos",          "Película",     91),
  ("U011", "Cosmos: Mundos Posibles",  "Documental",   47),
  ("U012", "La Casa de Papel",         "Serie",        53)
)

reproducciones: List[(String, String, String, Int)] = List(
  ("U001", "La Casa de Papel", "Serie", 47),
  ("U002", "Dune: Parte Dos", "Película", 95),
  ("U003", "Cosmos: Mundos Posibles", "Documental", 42),
  ("U001", "Dune: Parte Dos", "Película", 107),
  ("U004", "La Casa de Papel", "Serie", 52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie", 58),
  ("U002", "Cosmos: Mundos Posibles", "Documental", 38),
  ("U006", "La Casa de Papel", "Serie", 49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie", 61),
  ("U007", "Dune: Parte Dos", "Película", 88),
  ("U005", "La Casa de Papel", "Serie", 44),
  ("U008", "Cosmos: Mundos Posibles", "Documental", 55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie", 63),
  ("U009", "Dune: Parte Dos", "Película", 112),
  ("U006", "Cosmos: Mundos Posibles", "Documental", 29),
  ("U010", "La Casa de Papel", "Serie", 51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie", 57),
  ("U008", "La Casa de Papel", "Serie", 46),
  ("U001", "Cosmos: Mund

Tu misión: cuatro análisis para el informe de Carmen
Carmen necesita cuatro métricas concretas para su informe de tarde. Cada una es una tarea independiente que puedes resolver con lo que has aprendido hoy. Léelas todas antes de empezar para planificar el notebook.

Tarea 1 — Preparar el entorno y cargar los datos como RDD
Antes de poder analizar nada, necesitas poner en marcha el motor. Configura el entorno Spark completo: añade las dependencias con $ivy, silencia los logs, crea la SparkSession en modo local con un nombre de aplicación que identifique claramente el proyecto ("MediaStream-Analisis-Dia1"), y paraleliza la lista de reproducciones para convertirla en un RDD.

Una vez tengas el RDD, imprime su número de particiones y confirma la versión de Spark en uso. Carmen quiere ver que el entorno arrancó correctamente antes de confiar en los resultados.

Salida esperada al final de esta tarea:

✅ Entorno MediaStream listo
   Spark version : 4.1.1
   App name      : MediaStream-Analisis-Dia1
   Particiones   : (depende de tu CPU)
   Registros     : 25
Tarea 2 — ¿Cuántas reproducciones completas hubo ayer?
El equipo de producto define una "reproducción completa" como aquella en la que el usuario vio más de 45 minutos. Carmen quiere saber cuántas reproducciones del día anterior cumplen ese criterio y qué porcentaje representan sobre el total.

Utiliza una transformación filter seguida de la acción count() para obtener el dato. Calcula el porcentaje haciendo la división entre el conteo filtrado y el total de registros.

Salida esperada:

── Reproducciones completas (> 45 min) ──
Total de reproducciones ayer : 25
Reproducciones completas     : 18
Porcentaje completadas       : 72.0%
Tarea 3 — ¿Qué género consumió más minutos en total?
El departamento de contenidos necesita saber en qué género se concentró el consumo del día para tomar decisiones de inversión. Necesitas calcular el total de minutos vistos agrupado por género.

Para lograrlo, transforma el RDD en pares (genero, minutos) y luego usa reduceByKey para sumar los minutos de cada género. Ordena el resultado de mayor a menor consumo y recógelo con collect() para mostrarlo.

Salida esperada:

── Minutos totales por género ──
Serie       → 748 minutos
Película    → 595 minutos
Documental  → 244 minutos
Tarea 4 — ¿Cuál es el contenido más visto y cuántos usuarios únicos lo vieron?
Carmen quiere saber qué título acumuló más reproducciones durante el día y, de ese título, cuántos usuarios distintos lo vieron. Son dos preguntas relacionadas que puedes resolver en secuencia sobre el mismo RDD.

Para la primera parte, transforma el RDD en pares (titulo, 1) y usa reduceByKey para contar el número de reproducciones por título. Usa reduce sobre ese resultado para quedarte con el par que tenga el conteo más alto.

Para la segunda parte, filtra el RDD original para quedarte solo con las reproducciones del título más visto, extrae los identificadores de usuario con map, elimina los duplicados con distinct() y cuenta el resultado.

Salida esperada:

── Contenido más popular ──
Título más visto    : La Casa de Papel
Número de reproducciones : 8
Usuarios únicos que lo vieron : 7
🚀 Pregunta adicional
Carmen tiene una petición extra: identifica qué usuario individual acumuló más minutos totales de visualización en el día. Necesitarás transformar el RDD en pares (usuario_id, minutos), sumar los minutos por usuario con reduceByKey, y luego usar reduce para encontrar el par con el valor más alto.

Salida esperada del reto:

── Usuario más activo del día ──
Usuario : U001
Minutos totales vistos : 187
